# Union Budget Big Data Analytics
**Scope note:** this notebook analyzes preliminary extracted demand rows only. Values are not fully validated against all official PDFs.

## 1. Load data
Run `python code/restore_dataset.py` once before starting the notebook to reconstruct the CSV.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
csv_path = ROOT / 'data' / 'Union_Budget_Analytical_Dataset.csv'
spark = SparkSession.builder.appName('UnionBudgetAnalytics').getOrCreate()
df = spark.read.option('header', True).option('inferSchema', True).csv(str(csv_path))
df.printSchema()
df.show(5, truncate=False)

## 2. Data quality checks
Nulls are unknown/unverified amounts, not zero. Check these against source PDFs.

In [ ]:
numeric = ['Revenue_Cr', 'Capital_Cr', 'Total_Cr']
df.select([F.sum(F.col(c).isNull().cast('int')).alias(c + '_missing') for c in numeric]).show()
df.groupBy('Financial_Year', 'Demand_No').count().filter(F.col('count') > 1).show()
df.groupBy('Financial_Year').agg(F.count('*').alias('rows'), F.count('Total_Cr').alias('rows_with_total'), F.sum('Total_Cr').alias('recorded_total_cr')).orderBy('Financial_Year').show(truncate=False)

## 3. Department-level descriptive statistics

In [ ]:
df.groupBy('Demand_or_Department').agg(F.count('Total_Cr').alias('n'), F.min('Total_Cr').alias('min_cr'), F.max('Total_Cr').alias('max_cr'), F.avg('Total_Cr').alias('avg_cr'), F.stddev('Total_Cr').alias('stddev_cr')).orderBy(F.desc('max_cr')).show(20, truncate=False)

## 4. Recorded annual totals and change
These are totals of available extracted rows, not official consolidated budget totals. Years are not contiguous.

In [ ]:
annual = df.groupBy('Financial_Year').agg(F.sum('Total_Cr').alias('recorded_total_cr')).orderBy('Financial_Year')
w = Window.orderBy('Financial_Year')
annual = annual.withColumn('previous_recorded_total', F.lag('recorded_total_cr').over(w)).withColumn('change_cr', F.col('recorded_total_cr') - F.col('previous_recorded_total')).withColumn('change_pct', (F.col('recorded_total_cr') / F.col('previous_recorded_total') - 1) * 100)
annual.show(truncate=False)

In [ ]:
spark.stop()